# M4U3 PPE Detection — Inference (use the trained models)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/arqmanu/M4U3_ppe-detection-yolov8/blob/main/notebooks/02_Inference.ipynb)

This notebook is for **using** the models, not training them. It runs on CPU in about 1–2 minutes and needs no credentials.

1. Downloads the published weights from the GitHub Releases and verifies their SHA-256.
2. **Baseline:** runs the generic COCO model (`yolov8n.pt`), which has no PPE classes.
3. Runs the **trained PPE models** on the 5 new test images (never uploaded to Roboflow):
   - iteration 1 (2 PPE classes);
   - iteration 2 (adds the violation classes `no_head_protection` and `no_high_visibility_clothing`).
4. *(Optional)* Lets you upload **your own photo** and screen it.

Training and evaluation are in [`01_Training_Evaluation.ipynb`](01_Training_Evaluation.ipynb) (iteration 1) and [`03_Iteration2_Compliance.ipynb`](03_Iteration2_Compliance.ipynb) (iteration 2).

> **Safety disclaimer:** This model is an assistive tool for preliminary screening only. It produces false negatives and false positives. It must not be used as the sole verifier for life-safety decisions.

## 1. Environment and dependencies
Checks whether a GPU is available and installs the pinned Ultralytics version used for the reported results (`8.2.103`). It is installed without changing Colab's pre-installed libraries, so no runtime restart is needed.

In [ ]:
import os, time, platform
from datetime import datetime, timezone

T0 = time.time()                      # used by the reproducibility proof at the end
HOME = "/content"
os.makedirs(HOME, exist_ok=True)
os.chdir(HOME)

import torch
GPU_AVAILABLE = torch.cuda.is_available()
DEVICE_NAME = torch.cuda.get_device_name(0) if GPU_AVAILABLE else f"CPU ({platform.processor() or 'unknown'})"
print("GPU available:", GPU_AVAILABLE)
print("Device:", DEVICE_NAME)

In [ ]:
# Ultralytics 8.2.103 declares numpy<2, which would downgrade Colab's NumPy 2 and break pandas/matplotlib
# ("numpy.dtype size changed"). We therefore install it without touching Colab's pre-installed packages
# and add its two small missing dependencies.
%pip install -q --no-deps ultralytics==8.2.103
%pip install -q ultralytics-thop py-cpuinfo

import numpy as np
if not hasattr(np, "trapz"):          # NumPy >= 2 renamed trapz -> trapezoid (used by Ultralytics 8.2 mAP code)
    np.trapz = np.trapezoid

!yolo settings sync=False

import ultralytics
from ultralytics import YOLO
ultralytics.checks()

## 2. Download the published weights and the new test images
Each weights file is checked against the SHA-256 published in the README. If the hash does not match, the notebook stops.

In [ ]:
import hashlib, urllib.request, subprocess
from pathlib import Path
from PIL import Image as PILImage
import matplotlib.pyplot as plt
from IPython.display import display

WEIGHTS = {
    "iteration1": ("https://github.com/arqmanu/M4U3_ppe-detection-yolov8/releases/download/v1.0/best.pt",
                   "e72ead7d46303af3798b12a28af7d6dde4e222588761fbc63d7d6cd668ee6bc2"),
    "iteration2": ("https://github.com/arqmanu/M4U3_ppe-detection-yolov8/releases/download/v2.0/best.pt",
                   "dc70d91ca09e974c4ff3c62d1796f1929d077cc310c1b29ba12080112d37881f"),
}


def sha256_file(path, chunk=1 << 20):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(chunk), b""):
            h.update(block)
    return h.hexdigest()


models = {}
for name, (url, sha) in WEIGHTS.items():
    path = Path(f"/content/weights_{name}/best.pt")
    path.parent.mkdir(parents=True, exist_ok=True)
    if not path.exists():
        urllib.request.urlretrieve(url, path)
    assert sha256_file(path) == sha, f"Checksum mismatch for {name}"
    models[name] = YOLO(str(path))
    print(f"{name}: weights verified · classes = {list(models[name].names.values())}")

REPO_URL = "https://github.com/arqmanu/M4U3_ppe-detection-yolov8.git"
REPO_DIR = Path("/content/M4U3_repo")
if REPO_DIR.exists():
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "-q"], check=True)
else:
    subprocess.run(["git", "clone", "-q", REPO_URL, str(REPO_DIR)], check=True)
new_images = sorted((REPO_DIR / "results" / "03_new_test_images").glob("*.jpg"))
print(f"{len(new_images)} new test images:", [p.name for p in new_images])

OUT = Path("/content/outputs_inference")
OUT.mkdir(parents=True, exist_ok=True)

## 3. Baseline vs trained models on the new images
Rows: **COCO baseline** (can find `person`, but has no PPE classes) · **iteration 1** (valid PPE) · **iteration 2** (valid PPE + violations). Confidence threshold 0.25.

In [ ]:
runs = {"COCO baseline": YOLO("yolov8n.pt"), "Iteration 1": models["iteration1"], "Iteration 2": models["iteration2"]}
results = {k: m.predict(source=[str(p) for p in new_images], imgsz=640, conf=0.25, verbose=False) for k, m in runs.items()}

fig, axes = plt.subplots(len(runs), len(new_images), figsize=(3.6 * len(new_images), 6.4 * len(runs)))
for i, (k, res) in enumerate(results.items()):
    for j, (p, r) in enumerate(zip(new_images, res)):
        img = r.plot()[:, :, ::-1]
        axes[i, j].imshow(img); axes[i, j].axis("off")
        axes[i, j].set_title(f"{k} · {p.name}", fontsize=9)
        PILImage.fromarray(img).save(OUT / f"{k.replace(' ', '_')}_{p.name}")
plt.tight_layout(); plt.show()

for k, res in results.items():
    print(f"\n{k}")
    for p, r in zip(new_images, res):
        names = [runs[k].names[int(c)] for c in r.boxes.cls]
        print(f"  {p.name}: {', '.join(sorted(set(names))) or 'nothing detected'}")

## 4. *(Optional)* Screen your own photo
Set `UPLOAD_OWN_IMAGE = True` and run the cell. Colab will ask you to choose a photo from your computer. With `False` (the default), the cell is skipped, so `Run all` never stops to wait for an upload.

In [ ]:
UPLOAD_OWN_IMAGE = False
MODEL_FOR_UPLOAD = "iteration2"      # "iteration1" or "iteration2"

if UPLOAD_OWN_IMAGE:
    from google.colab import files
    uploaded = files.upload()
    for fname in uploaded:
        r = models[MODEL_FOR_UPLOAD].predict(source=fname, imgsz=640, conf=0.25, verbose=False)[0]
        display(PILImage.fromarray(r.plot()[:, :, ::-1]).resize((480, int(480 * r.orig_shape[0] / r.orig_shape[1]))))
        flags = [models[MODEL_FOR_UPLOAD].names[int(c)] for c in r.boxes.cls]
        violations = [f for f in flags if f.startswith("no_")]
        print("Detections:", flags or "none")
        if violations:
            print("⚠ Possible PPE violation, to be verified by a person:", violations)
else:
    print("Upload skipped (set UPLOAD_OWN_IMAGE = True to try your own photo).")

## 5. Summary
All predictions are saved in `/content/outputs_inference`. Remember that these results are a **pre-screening**: every image must be reviewed by a competent person.

In [ ]:
import sys
print(f"Inference completed in {(time.time() - T0) / 60:.1f} min on {DEVICE_NAME} "
      f"(Python {sys.version.split()[0]}, PyTorch {torch.__version__}, Ultralytics {ultralytics.__version__}).")
print(len(list(OUT.glob('*.jpg'))), "prediction images saved to", OUT)